In [5]:
# Standard Library
import os, sys, json, math, time, csv, shutil, traceback
from pathlib import Path

# Third-Party
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import kagglehub
from scipy import stats as scipy_stats
from scipy.ndimage import gaussian_filter1d
from scipy.ndimage import label as nd_label
from ultralytics import YOLO

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# Sklearn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, matthews_corrcoef,
    balanced_accuracy_score, cohen_kappa_score,
    average_precision_score, precision_recall_curve,
)
import seaborn as sns

print("\u2705 Imports complete.")


# ── Output paths ──────────────────────────────────────────────────────────
STAGE1_DIR   = "/content/stage1_output"
STAGE2_DIR   = "/content/stage2_output"
STAGE3_DIR   = "/content/stage3_output"   # cumulative JSON (s1+s2+s3 per video)
STAGE4_DIR   = "/content/stage4_output"   # stage3 JSON + stage4 block appended
SEQUENCE_CSV = "/content/sequence_features.csv"   # per-frame BiLSTM input
RESULTS_CSV  = "/content/results.csv"             # one row per video, all scores

VIS_ROOT = "/content/visualisation"
VIS_S1   = os.path.join(VIS_ROOT, "stage1")
VIS_S2   = os.path.join(VIS_ROOT, "stage2")
VIS_S3   = os.path.join(VIS_ROOT, "stage3")

for d in [STAGE1_DIR, STAGE2_DIR, STAGE3_DIR, STAGE4_DIR, VIS_S1, VIS_S2, VIS_S3]:
    os.makedirs(d, exist_ok=True)

print("\u2705 Directories ready.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Imports complete.
✅ Directories ready.


In [7]:
# ── Constants ─────────────────────────────────────────────────────────────
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLIP_LEN   = 32
FRAME_SIZE = 224
NUM_KP     = 17
YOLO_IMGSZ = 640
YOLO_CONF  = 0.1
YOLO_IOU   = 0.5

FEATURE_COLUMNS = [
    "max_depth_drop_ratio",
    "height_drop_ratio",
    "peak_time_ratio",
    "ground_proximity",
    "peak_motion",
    "tilt_angle",
    "post_impact_stillness",
    "depth_peak_time_ratio",
    "tilt_relative_change",
    "motion_energy",
    "acceleration_energy",
    "duration_high_motion",
]

INPUT_DIM  = 12    # len(FEATURE_COLUMNS)
HIDDEN_DIM = 64

SKELETON_LINES = [
    (5,6),(5,11),(6,12),(11,12),
    (5,7),(7,9),(6,8),(8,10),
    (11,13),(13,15),(12,14),(14,16),
    (0,1),(0,2),(1,3),(2,4),
]


# ── Stage-1  3D-CNN ────────────────────────────────────────────────────────
class Stage1_3DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv3d(3, 64, (3,7,7), (1,2,2), (1,3,3), padding_mode="replicate")
        self.bn1   = nn.BatchNorm3d(64)
        self.pool1 = nn.MaxPool3d((1,3,3), (1,2,2), (0,1,1))
        self.conv2 = nn.Conv3d(64, 128, 3, padding=1, padding_mode="replicate")
        self.bn2   = nn.BatchNorm3d(128)
        self.pool2 = nn.MaxPool3d(2, 2)
        self.conv3 = nn.Conv3d(128, 256, 3, padding=1, padding_mode="replicate")
        self.bn3   = nn.BatchNorm3d(256)
        self.conv4 = nn.Conv3d(256, 256, 3, padding=1, padding_mode="replicate")
        self.bn4   = nn.BatchNorm3d(256)
        self.global_pool = nn.AdaptiveAvgPool3d((1,1,1))
        self.dropout     = nn.Dropout(0.5)
        self.fc          = nn.Linear(256, 2)

    def forward(self, x, return_saliency=False):
        x   = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x   = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x   = F.relu(self.bn3(self.conv3(x)))
        sal = x.detach().pow(2).mean(dim=(1, 3, 4))
        T_p = sal.shape[1]
        em  = torch.ones(T_p, device=sal.device)
        em[0] = 0.05;  em[-1] = 0.5
        sal = sal * em
        sal = (sal - sal.min(1,keepdim=True)[0]) / (
              sal.max(1,keepdim=True)[0] - sal.min(1,keepdim=True)[0] + 1e-6)
        sal = F.softmax(sal * 5.0, dim=1)
        x   = F.relu(self.bn4(self.conv4(x)))
        x   = self.global_pool(x).flatten(1)
        x   = self.dropout(x)
        logits = self.fc(x)
        if return_saliency:
            return logits, sal
        return logits


# ── Checkpoint paths — UPDATE THESE BEFORE RUNNING ──────────────────────
STAGE1_CKPT = "/content/stage1.pth"

# Folder containing:
#   model_bed.pth  model_chair.pth  model_stand.pth
#   scaler_bed.pkl scaler_chair.pkl scaler_stand.pkl
#   config_bed.json config_chair.json config_stand.json
BILSTM_MODEL_DIR = "/content/bilstm_models"

STAGE5_PERCENTILE_FILE = "/content/stage5_percentiles.json"

# ── Stage-1 ───────────────────────────────────────────────────────────────
stage1_model = Stage1_3DCNN().to(DEVICE)
stage1_model.load_state_dict(torch.load(STAGE1_CKPT, map_location=DEVICE))
stage1_model.eval()
print("\u2705 Stage-1 3D-CNN loaded")

# ── MiDaS (Stage-3) ────────────────────────────────────────────────────────
print("Loading MiDaS...")
midas           = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas_transform = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform
midas.to(DEVICE).eval()
print("\u2705 MiDaS loaded")

# ── YOLO pose (Stage-2) ────────────────────────────────────────────────────
yolo_model = YOLO("yolov8m-pose.pt")
print("\u2705 YOLOv8 pose loaded")


# ═══════════════════════════════════════════════════════════════════════════
# HELPER — exhaustive temporal feature extractor
# Used by Stage-2 (pose signals) and Stage-3 (depth signal)
# ═══════════════════════════════════════════════════════════════════════════
def all_temporal_features(values, prefix, saliency_weights=None):
    arr = np.array(values, dtype=np.float64)
    n   = len(arr)

    def r(v):
        v = float(v)
        return 0.0 if (math.isnan(v) or math.isinf(v)) else round(v, 6)

    ZERO_KEYS = [
        "max","min","mean","std","median","range","sum",
        "delta","abs_delta","norm_delta","linear_slope",
        "peak_pos","trough_pos","time_to_peak","peak_trough_gap",
        "pre_peak_slope","post_peak_slope","slope_ratio","curvature",
        "first_half_mean","second_half_mean","first_half_std","second_half_std",
        "half_mean_diff","half_mean_ratio",
        "first_third_mean","mid_third_mean","last_third_mean",
        "energy","skewness","kurtosis","iqr",
        "mean_crossings","zero_crossings","above_mean_ratio",
        "mean_abs_diff","max_diff","total_variation","rms_diff",
    ]
    if n == 0:
        out = {f"{prefix}_{k}": 0.0 for k in ZERO_KEYS}
        if saliency_weights is not None:
            for k in ["weighted_mean","weighted_std","weighted_peak_val","weighted_sum"]:
                out[f"{prefix}_{k}"] = 0.0
        return out

    sig_range  = float(np.max(arr) - np.min(arr))
    delta      = float(arr[-1] - arr[0])
    norm_delta = delta / sig_range if sig_range > 1e-9 else 0.0
    lin_coeff  = np.polyfit(np.arange(n), arr, 1)[0] if n > 1 else 0.0
    peak_idx   = int(np.argmax(arr))
    trough_idx = int(np.argmin(arr))
    peak_pos   = peak_idx   / max(n - 1, 1)
    trough_pos = trough_idx / max(n - 1, 1)
    pre_sl  = float(np.mean(np.diff(arr[:peak_idx+1]))) if peak_idx > 0     else 0.0
    post_sl = float(np.mean(np.diff(arr[peak_idx:])))   if peak_idx < n - 1 else 0.0
    curvature = np.polyfit(np.arange(n), arr, 2)[0] if n >= 3 else 0.0
    mid  = max(n // 2, 1)
    fh   = arr[:mid]
    sh   = arr[mid:] if mid < n else arr[-1:]
    t1   = max(n // 3, 1)
    t2   = max(2 * n // 3, t1 + 1)
    mean_v   = float(np.mean(arr))
    centered = arr - mean_v
    if n > 1:
        diffs = np.abs(np.diff(arr))
        mean_ad = float(np.mean(diffs));  max_d   = float(np.max(diffs))
        tot_var = float(np.sum(diffs));   rms_d   = float(np.sqrt(np.mean(diffs**2)))
    else:
        mean_ad = max_d = tot_var = rms_d = 0.0

    out = {
        f"{prefix}_max":              r(np.max(arr)),
        f"{prefix}_min":              r(np.min(arr)),
        f"{prefix}_mean":             r(np.mean(arr)),
        f"{prefix}_std":              r(np.std(arr)),
        f"{prefix}_median":           r(np.median(arr)),
        f"{prefix}_range":            r(sig_range),
        f"{prefix}_sum":              r(np.sum(arr)),
        f"{prefix}_delta":            r(delta),
        f"{prefix}_abs_delta":        r(abs(delta)),
        f"{prefix}_norm_delta":       r(norm_delta),
        f"{prefix}_linear_slope":     r(lin_coeff),
        f"{prefix}_peak_pos":         r(peak_pos),
        f"{prefix}_trough_pos":       r(trough_pos),
        f"{prefix}_time_to_peak":     r(peak_pos),
        f"{prefix}_peak_trough_gap":  r(abs(peak_pos - trough_pos)),
        f"{prefix}_pre_peak_slope":   r(pre_sl),
        f"{prefix}_post_peak_slope":  r(post_sl),
        f"{prefix}_slope_ratio":      r(pre_sl / (post_sl + 1e-9)),
        f"{prefix}_curvature":        r(curvature),
        f"{prefix}_first_half_mean":  r(float(np.mean(fh))),
        f"{prefix}_second_half_mean": r(float(np.mean(sh))),
        f"{prefix}_first_half_std":   r(float(np.std(fh))),
        f"{prefix}_second_half_std":  r(float(np.std(sh))),
        f"{prefix}_half_mean_diff":   r(float(np.mean(sh)) - float(np.mean(fh))),
        f"{prefix}_half_mean_ratio":  r(float(np.mean(sh)) / (float(np.mean(fh)) + 1e-9)),
        f"{prefix}_first_third_mean": r(float(np.mean(arr[:t1]))),
        f"{prefix}_mid_third_mean":   r(float(np.mean(arr[t1:t2]))),
        f"{prefix}_last_third_mean":  r(float(np.mean(arr[t2:])) if t2 < n else float(arr[-1])),
        f"{prefix}_energy":           r(float(np.sum(arr**2))),
        f"{prefix}_skewness":         r(float(scipy_stats.skew(arr))     if n >= 3 else 0.0),
        f"{prefix}_kurtosis":         r(float(scipy_stats.kurtosis(arr)) if n >= 4 else 0.0),
        f"{prefix}_iqr":              r(float(np.percentile(arr,75) - np.percentile(arr,25))),
        f"{prefix}_mean_crossings":   r(int(np.sum(np.diff(np.sign(centered)) != 0))),
        f"{prefix}_zero_crossings":   r(int(np.sum(np.diff(np.sign(arr)) != 0))),
        f"{prefix}_above_mean_ratio": r(float(np.sum(arr > mean_v) / n)),
        f"{prefix}_mean_abs_diff":    r(mean_ad),
        f"{prefix}_max_diff":         r(max_d),
        f"{prefix}_total_variation":  r(tot_var),
        f"{prefix}_rms_diff":         r(rms_d),
    }

    if saliency_weights is not None:
        w      = np.array(saliency_weights, dtype=np.float64)[:n]
        w      = w / (w.sum() + 1e-9)
        w_mean = float(np.sum(arr * w))
        com    = max(0, min(int(np.round(np.sum(np.arange(n) * w))), n - 1))
        out.update({
            f"{prefix}_weighted_mean":     r(w_mean),
            f"{prefix}_weighted_std":      r(float(np.sqrt(np.sum(w * (arr - w_mean)**2)))),
            f"{prefix}_weighted_peak_val": r(float(arr[com])),
            f"{prefix}_weighted_sum":      r(float(np.sum(arr * w))),
        })

    return out


print("\u2705 Temporal feature extractor defined.")


# ═══════════════════════════════════════════════════════════════════════════
# STAGE 1 — 3D-CNN Saliency
# ═══════════════════════════════════════════════════════════════════════════
def _get_active_window(saliency, energy_threshold=0.15):
    thresh    = np.mean(saliency) + 0.5 * np.std(saliency)
    is_active = saliency > thresh
    labs, nf  = nd_label(is_active)
    if nf == 0:
        return 0, len(saliency) - 1, "Static/Uniform"
    total_e = np.sum(saliency)
    sig_idx = []
    for i in range(1, nf + 1):
        m = labs == i
        if np.sum(saliency[m]) / total_e > energy_threshold:
            sig_idx.extend(np.where(m)[0].tolist())
    if not sig_idx:
        pk = int(np.argmax(saliency))
        return max(0, pk - 2), min(len(saliency) - 1, pk + 2), "Peak Only"
    return int(min(sig_idx)), int(max(sig_idx)), "Global Event Zone"


def run_stage1(folder_name, image_paths):
    # image_paths: sorted list of exactly 32 uniformly-sampled images.
    # clip_indices[i] = i  (position i == image_paths index i)
    frames = []
    for p in image_paths:
        img = cv2.imread(p)
        img = cv2.resize(img, (FRAME_SIZE, FRAME_SIZE))
        frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    frames_np = np.array(frames, dtype=np.float32) / 255.0
    clip = torch.from_numpy(frames_np).permute(3,0,1,2).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        _, saliency = stage1_model(clip, return_saliency=True)

    sal_raw    = saliency.squeeze(0).cpu().numpy()
    sal_interp = np.interp(
        np.linspace(0, len(sal_raw)-1, CLIP_LEN),
        np.arange(len(sal_raw)), sal_raw
    )
    sal_smooth = gaussian_filter1d(sal_interp, sigma=1.2)
    start, end, status = _get_active_window(sal_smooth)

    meta = {
        "active_window":    [int(start), int(end)],
        "varying_k":        int(end - start + 1),
        "detection_status": status,
        "saliency_weights": sal_smooth.tolist(),
        "clip_indices":     list(range(CLIP_LEN)),
    }

    with open(os.path.join(STAGE1_DIR, f"{folder_name}.json"), "w") as fh:
        json.dump({"folder_name": folder_name, **meta}, fh, indent=4)

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(sal_interp, alpha=0.5, label="Raw")
    ax.plot(sal_smooth, lw=2,      label="Smoothed")
    ax.axvspan(start, end, alpha=0.2, color="red", label=f"Active [{start},{end}]")
    ax.set_title(f"Stage-1 Saliency | {folder_name}")
    ax.legend(); ax.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(VIS_S1, f"{folder_name}.png"), dpi=80)
    plt.close()

    return meta


print("\u2705 Stage-1 defined.")


# ═══════════════════════════════════════════════════════════════════════════
# STAGE 2 — YOLO Pose + Physics Features + Stage-2 Summary
# ═══════════════════════════════════════════════════════════════════════════
def _norm_kp(kp_px, W, H):
    if kp_px is None:
        return [[0.0, 0.0]] * NUM_KP
    return [[float(x / W), float(y / H)] for x, y in kp_px]


def _interpolate_kp(frame_nums, det_norm):
    L  = len(frame_nums)
    xs = np.full((L, NUM_KP), np.nan)
    ys = np.full((L, NUM_KP), np.nan)
    for i, f in enumerate(frame_nums):
        for j, pt in enumerate(det_norm[f]):
            if pt[0] is not None:
                xs[i, j] = pt[0]
                ys[i, j] = pt[1]
    idx = np.arange(L)
    for j in range(NUM_KP):
        v = ~np.isnan(xs[:, j])
        xs[:, j] = np.interp(idx, idx[v], xs[v, j]) if v.any() else 0.0
        ys[:, j] = np.interp(idx, idx[v], ys[v, j]) if v.any() else 0.0
    return {
        f: [[float(xs[i,j]), float(ys[i,j])] for j in range(NUM_KP)]
        for i, f in enumerate(frame_nums)
    }


def run_stage2(folder_name, stage1_meta, image_paths):
    # Returns (frame_data, s2_summary, detected).
    # torso_depth not yet in frame_data features — added in-place by Stage-3.
    win_s      = stage1_meta["active_window"][0]
    win_e      = stage1_meta["active_window"][1]
    frame_nums = list(range(win_s, win_e + 1))

    raw_frames = {f: cv2.imread(image_paths[f]) for f in frame_nums}
    ref        = raw_frames[frame_nums[0]]
    H0, W0     = ref.shape[:2]

    det_px   = {}
    bbox_px  = {}
    scores   = {}
    detected = {}

    vis_tmp = os.path.join(VIS_S2, f"_tmp_{folder_name}")
    os.makedirs(vis_tmp, exist_ok=True)

    for fnum in frame_nums:
        frame   = raw_frames[fnum]
        results = yolo_model.predict(
            source=frame, imgsz=YOLO_IMGSZ, conf=YOLO_CONF, iou=YOLO_IOU, verbose=False
        )
        vis = frame.copy()
        if results and len(results[0].boxes) > 0:
            best           = results[0].boxes.conf.argmax()
            kp             = results[0].keypoints.data[best].cpu().numpy()
            bb             = results[0].boxes.xyxy[best].cpu().numpy()
            det_px[fnum]   = kp[:, :2]
            scores[fnum]   = float(results[0].boxes.conf[best])
            bbox_px[fnum]  = [
                float((bb[0]+bb[2])/2), float((bb[1]+bb[3])/2),
                float(bb[2]-bb[0]),     float(bb[3]-bb[1])
            ]
            detected[fnum] = True
            for s_j, e_j in SKELETON_LINES:
                p1, p2 = kp[s_j], kp[e_j]
                if p1[2] > 0.3 and p2[2] > 0.3:
                    cv2.line(vis,
                             (int(p1[0]),int(p1[1])),
                             (int(p2[0]),int(p2[1])), (255,255,0), 2)
            cv2.rectangle(vis,
                          (int(bb[0]),int(bb[1])),
                          (int(bb[2]),int(bb[3])), (0,0,255), 2)
        else:
            det_px[fnum]   = None
            bbox_px[fnum]  = None
            scores[fnum]   = 0.0
            detected[fnum] = False
        cv2.imwrite(os.path.join(vis_tmp, f"frame_{fnum:03d}.png"), vis)

    det_norm  = {f: _norm_kp(det_px[f], W0, H0) for f in frame_nums}
    bbox_norm = {
        f: [bbox_px[f][0]/W0, bbox_px[f][1]/H0,
            bbox_px[f][2]/W0, bbox_px[f][3]/H0]
           if bbox_px[f] else [None]*4
        for f in frame_nums
    }
    interp = _interpolate_kp(frame_nums, det_norm)

    frame_data = []
    prev_hip   = None
    for fnum in frame_nums:
        kp   = interp[fnum]
        m_sh = [(kp[5][0]+kp[6][0])/2, (kp[5][1]+kp[6][1])/2]
        m_hp = [(kp[11][0]+kp[12][0])/2, (kp[11][1]+kp[12][1])/2]
        tilt = abs(math.degrees(math.atan2(m_sh[0]-m_hp[0], -(m_sh[1]-m_hp[1]))))
        vel  = float(m_hp[1] - prev_hip[1]) if prev_hip else 0.0
        prev_hip = m_hp
        bx   = bbox_norm[fnum]
        hw   = float(bx[3]/bx[2]) if (bx[2] and bx[2] > 0) else 0.0
        gp   = float(1.0 - m_sh[1])
        frame_data.append({
            "frame_idx":       int(fnum),
            "keypoints":       kp,
            "normalized_bbox": bx,
            "score":           round(scores[fnum], 6),
            "features": {
                "tilt_angle":        round(tilt, 6),
                "vertical_velocity": round(vel,  6),
                "h_w_ratio":         round(hw,   6),
                "ground_proximity":  round(gp,   6),
                # torso_depth added in-place by Stage-3
            },
        })

    sal_window = np.array(stage1_meta["saliency_weights"])[win_s:win_e + 1]
    s2_summary = {}
    s2_summary.update(all_temporal_features(
        [fd["features"]["tilt_angle"]        for fd in frame_data],
        "tilt", saliency_weights=sal_window))
    s2_summary.update(all_temporal_features(
        [fd["features"]["h_w_ratio"]         for fd in frame_data],
        "h_w_ratio", saliency_weights=sal_window))
    s2_summary.update(all_temporal_features(
        [fd["features"]["vertical_velocity"] for fd in frame_data],
        "velocity", saliency_weights=sal_window))
    s2_summary.update(all_temporal_features(
        [fd["features"]["ground_proximity"]  for fd in frame_data],
        "ground_proximity", saliency_weights=sal_window))

    # Stage-2 JSON (frame_data without torso_depth — written for reference)
    with open(os.path.join(STAGE2_DIR, f"{folder_name}.json"), "w") as fh:
        json.dump({"frame_data": frame_data}, fh, indent=2)

    # Skeleton montage
    skel_imgs = []
    for fnum in frame_nums:
        p = os.path.join(vis_tmp, f"frame_{fnum:03d}.png")
        if os.path.exists(p):
            skel_imgs.append(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    if skel_imgs:
        cols = min(4, len(skel_imgs))
        rows = math.ceil(len(skel_imgs) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
        axes = np.array(axes).flatten()
        for ai, si in enumerate(skel_imgs):
            axes[ai].imshow(si)
            axes[ai].set_title(f"F{frame_nums[ai]}", fontsize=8)
            axes[ai].axis("off")
        for ax in axes[len(skel_imgs):]:
            ax.axis("off")
        plt.suptitle(f"Stage-2 | {folder_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(VIS_S2, f"{folder_name}.png"), dpi=80)
        plt.close()

    return frame_data, s2_summary, detected


print("\u2705 Stage-2 defined.")


# ═══════════════════════════════════════════════════════════════════════════
# STAGE 3 — MiDaS Depth + Stage-3 Summary + Cumulative JSON
# ═══════════════════════════════════════════════════════════════════════════
def _torso_centroid(keypoints, H, W):
    ids = [5, 6, 11, 12]
    return (
        int(np.median([keypoints[t][0] * W for t in ids])),
        int(np.median([keypoints[t][1] * H for t in ids])),
    )


def run_stage3(folder_name, label_int, label_str,
               stage1_meta, s2_summary, frame_data, detected, image_paths):
    # s2_summary is passed explicitly as an argument (no closure dependency).
    # Mutates frame_data in-place: adds torso_depth to each frame features.
    # Writes cumulative Stage-3 JSON; returns (s3_summary, cumulative_dict).
    win_s      = stage1_meta["active_window"][0]
    win_e      = stage1_meta["active_window"][1]
    sal_window = np.array(stage1_meta["saliency_weights"])[win_s:win_e + 1]
    depth_vals = []

    for fd in frame_data:
        fnum = fd["frame_idx"]
        bgr  = cv2.imread(image_paths[fnum])
        H, W = bgr.shape[:2]
        rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        inp  = midas_transform(rgb).to(DEVICE)
        with torch.no_grad():
            pred = midas(inp)
            pred = F.interpolate(
                pred.unsqueeze(1), size=rgb.shape[:2],
                mode="bicubic", align_corners=False
            ).squeeze()
        d    = pred.cpu().numpy()
        dmap = (d - d.min()) / (d.max() - d.min() + 1e-6)

        if detected.get(fnum, False):
            cx, cy = _torso_centroid(fd["keypoints"], H, W)
        else:
            cx, cy = W // 2, H // 2

        cx    = max(0, min(cx, W - 1))
        cy    = max(0, min(cy, H - 1))
        d_val = float(dmap[cy, cx])
        fd["features"]["torso_depth"] = round(d_val, 6)
        depth_vals.append(d_val)

    s3_summary = all_temporal_features(depth_vals, "depth", saliency_weights=sal_window)
    d_arr = np.array(depth_vals, dtype=np.float64)
    s3_summary["depth_drop"]     = round(float(np.mean(d_arr[:-1]) - d_arr[-1]), 6) if len(d_arr) > 1 else 0.0
    s3_summary["depth_variance"] = round(float(np.var(d_arr)), 6)

    video_id   = f"{label_str}/unknown/unknown/{folder_name}"
    cumulative = {
        "video_name":       folder_name,
        "video_id":         video_id,
        "label":            label_int,
        "label_str":        label_str,
        "active_window":    stage1_meta["active_window"],
        "varying_k":        stage1_meta["varying_k"],
        "detection_status": stage1_meta["detection_status"],
        "saliency_weights": stage1_meta["saliency_weights"],
        "clip_indices":     stage1_meta["clip_indices"],
        "frame_data":       frame_data,
        "stage2_summary":   s2_summary,
        "stage3_summary":   s3_summary,
    }

    with open(os.path.join(STAGE3_DIR, f"{folder_name}.json"), "w") as fh:
        json.dump(cumulative, fh, indent=2)

    # Depth visualisation (4 sample frames)
    sample_idxs = np.linspace(0, len(frame_data)-1, min(4, len(frame_data)), dtype=int)
    fig, axes   = plt.subplots(1, len(sample_idxs), figsize=(5*len(sample_idxs), 4))
    axes = np.array(axes).flatten()
    for ai, si in enumerate(sample_idxs):
        fd  = frame_data[si]
        bgr = cv2.imread(image_paths[fd["frame_idx"]])
        axes[ai].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        axes[ai].set_title(
            f"F{fd['frame_idx']}\nd={fd['features']['torso_depth']:.3f}", fontsize=8)
        axes[ai].axis("off")
    plt.suptitle(f"Stage-3 Depth | {folder_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(VIS_S3, f"{folder_name}.png"), dpi=80)
    plt.close()

    return s3_summary, cumulative


print("\u2705 Stage-3 defined.")



✅ Stage-1 3D-CNN loaded
Loading MiDaS...


/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip
Loading weights:  None
Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.com/isl-org/MiDaS/releases/download/v2_1/midas_v21_small_256.pt" to /root/.cache/torch/hub/checkpoints/midas_v21_small_256.pt


100%|██████████| 81.8M/81.8M [00:01<00:00, 74.3MB/s]
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


✅ MiDaS loaded
✅ YOLOv8 pose loaded
✅ Temporal feature extractor defined.
✅ Stage-1 defined.
✅ Stage-2 defined.
✅ Stage-3 defined.


In [9]:
# ===============================================================
# COMPLETE CAUCAFALL → STAGE1-3 → JSON → STAGE4 FINETUNE
# ===============================================================


import os
import cv2
import json
import torch
import joblib
import shutil
import zipfile
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import kagglehub

# pathCF = kagglehub.dataset_download("tuyenldvn/caucafall")
# print("Path to dataset files:", pathCF)

# #-------------------------------------------------------------------------------
# # Copying datasets into content/ from session's cache/
# #-------------------------------------------------------------------------------

# # Source directories
# CFALL_SRC = "/root/.cache/kagglehub/datasets/tuyenldvn/caucafall/versions/2/Dataset CAUCAFall/CAUCAFall"
# # Target directories
CFALL_DST = "/content/CAUCAFall"

# def move_dataset(src, dst):
#     if os.path.exists(dst):
#         print(f"{dst} already exists. Skipping copy.")
#     else:
#         shutil.copytree(src, dst)
#         print(f"Copied dataset to {dst}")

# # Move datasets
# move_dataset(CFALL_SRC, CFALL_DST)


# Set paths
RAW_PATH = CFALL_DST
DATA_ROOT = "/content"
FRAME_DIR = os.path.join(DATA_ROOT, "caucafall_32")
JSON_DIR  = os.path.join(DATA_ROOT, "caucafall_stage3_json")

# os.makedirs(FRAME_DIR, exist_ok=True)
# os.makedirs(JSON_DIR, exist_ok=True)


# # -----------------------------
# # 1. Label Mapping
# # -----------------------------
# FALL_CLASSES = [
#     "FallBackward",
#     "FallRight",
#     "FallLeft",
#     "FallForward"
# ]

def get_label(filename):
    return 1 if any(f in filename for f in FALL_CLASSES) else 0

# # -----------------------------
# # 2. Extract 32 Frames
# # -----------------------------
# print("Extracting frames...")

# video_list = []

# for root, _, files in os.walk(RAW_PATH):
#     for f in files:
#         if f.endswith(".avi"):
#             full_path = os.path.join(root, f)
#             video_list.append((full_path, get_label(f)))

# for video_path, label in tqdm(video_list):

#     video_name = os.path.splitext(os.path.basename(video_path))[0]
#     save_path  = os.path.join(FRAME_DIR, video_name)
#     os.makedirs(save_path, exist_ok=True)

#     cap = cv2.VideoCapture(video_path)
#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

#     if total_frames < CLIP_LEN:
#         cap.release()
#         continue

#     indices = np.linspace(0, total_frames - 1, CLIP_LEN).astype(int)

#     for i, idx in enumerate(indices):
#         cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
#         ret, frame = cap.read()
#         if not ret:
#             break
#         cv2.imwrite(os.path.join(save_path, f"{i:04d}.jpg"), frame)

#     cap.release()

# print("Frame extraction done.")

# # -----------------------------
# # 3. Run Stage1 → Stage3
# # -----------------------------
# print("Running Stage1-3...")

# for video_path, label in tqdm(video_list):

#     video_name = os.path.splitext(os.path.basename(video_path))[0]
#     frame_path = os.path.join(FRAME_DIR, video_name)

#     image_paths = sorted(glob(os.path.join(frame_path, "*.jpg")))
#     if len(image_paths) < CLIP_LEN:
#         continue

#     # Stage1
#     s1_meta = run_stage1(video_name, image_paths)

#     # Stage2
#     frame_data, s2_summary, detected = run_stage2(
#         video_name, s1_meta, image_paths
#     )

#     # Stage3
#     s3_summary, cumulative = run_stage3(
#         video_name,
#         label,
#         "fall" if label == 1 else "nonfall",
#         s1_meta,
#         s2_summary,
#         frame_data,
#         detected,
#         image_paths
#     )

#     # Save cumulative JSON
#     with open(os.path.join(JSON_DIR, video_name + ".json"), "w") as f:
#         json.dump(cumulative, f)

# print("Stage3 JSON saved.")

# -----------------------------
# 4. Build Feature Matrix
# -----------------------------
print("Building feature matrix...")

scaler = joblib.load("stage4_scaler.pkl")

X = []
y = []

for jf in glob(os.path.join(JSON_DIR, "*.json")):

    with open(jf, "r") as f:
        data = json.load(f)

    combined = {}
    combined.update(data["stage2_summary"])
    combined.update(data["stage3_summary"])

    if hasattr(scaler, "feature_names_in_"):
        vec = np.array(
            [combined[k] for k in scaler.feature_names_in_],
            dtype=np.float32
        )
    else:
        keys = sorted(combined.keys())
        vec = np.array([combined[k] for k in keys], dtype=np.float32)

    X.append(vec)
    y.append(get_label(os.path.basename(jf)))

X = scaler.transform(np.array(X))
y = np.array(y)

print("Feature matrix shape:", X.shape)

# -----------------------------
# 5. Load Stage4 Model
# -----------------------------
stage4_model = torch.load("model.pth", map_location=DEVICE)
stage4_model.to(DEVICE)
stage4_model.train()

# Loss selection safety
if isinstance(list(stage4_model.modules())[-1], torch.nn.Sigmoid):
    criterion = torch.nn.BCELoss()
else:
    criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(stage4_model.parameters(), lr=1e-5)

# -----------------------------
# 6. Fine-Tune
# -----------------------------
EPOCHS = 10
metrics = []

print("Starting fine-tuning...")

for epoch in range(EPOCHS):

    epoch_loss = 0
    all_probs = []
    all_preds = []
    all_labels = []

    for i in range(len(X)):

        x_tensor = torch.tensor(X[i]).unsqueeze(0).to(DEVICE)
        label_tensor = torch.tensor([y[i]], dtype=torch.float32).to(DEVICE)

        optimizer.zero_grad()
        output = stage4_model(x_tensor).squeeze()

        loss = criterion(output, label_tensor)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        prob = torch.sigmoid(output).item() if isinstance(criterion, torch.nn.BCEWithLogitsLoss) else output.item()
        pred = 1 if prob > 0.5 else 0

        all_probs.append(prob)
        all_preds.append(pred)
        all_labels.append(y[i])

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("Loss:", round(epoch_loss,4),
          "Acc:", round(acc,4),
          "Prec:", round(prec,4),
          "Rec:", round(rec,4),
          "F1:", round(f1,4),
          "AUC:", round(auc,4))

    metrics.append({
        "epoch": epoch+1,
        "loss": epoch_loss,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc
    })

# -----------------------------
# 7. Save Outputs
# -----------------------------
torch.save(stage4_model, "/content/stage4_finetuned_caucafall.pth")
pd.DataFrame(metrics).to_csv("/content/caucafall_finetune_metrics.csv", index=False)

print("\nPIPELINE COMPLETE.")

Building feature matrix...


KeyError: 'max_depth_drop_ratio'